### READ ME

Use the code blocks below to answer each question. Only print the output required for each question. Do not edit the comments at the top of each code cell. Otherwise, the auto-grader may misinterpret your results. See Question 0 as an example of how to complete a task (leave it in your notebook; don't delete it).

**Important: Working with DataFrames**

Throughout this assignment, you will progressively clean and transform data. Here's how to think about which DataFrame to use:

- **Questions 1-7**: Work with the main DataFrame `df` that you'll progressively clean (handling missing values, dates, binning, etc.)
- **Questions 8-15**: Create a NEW DataFrame called `df_model` (a copy of `df`) for the two-class model
- **Questions 16-19**: Continue using the original `df` to prepare for multi-class modeling
- **Questions 20-27**: Create a NEW DataFrame called `df_model` (a copy of the cleaned `df`) for the multi-class model

**General Rule**: If a question explicitly says "create a copy" or "create a new DataFrame," do it. Otherwise, continue working with your existing DataFrame. Each modeling section (two-class and multi-class) needs its own `df_model` because they use different labels.

In [136]:
# from google.colab import drive
# drive.mount('/content/drive')

### Question 0

Create a DataFrame with three rows and four columns. Name the columns 'Col1', 'Col2', 'Col3', 'Col4'. Create an index for the DataFrame and give the rows the index values of 'Row1', 'Row2', 'Row3'. Place a value in each column equal to the {ColumnName/RowName}. e.g. Col1/Row1. Print the entire DataFrame.

In [137]:
# Question 0

import pandas as pd

df = pd.DataFrame(columns=['Col1', 'Col2', 'Col3', 'Col4'], index=['Row1', 'Row2', 'Row3'])

for col in df:
  for i, value in df[col].items():
    df.at[i, col] = f'{i}/{col}'

df

,Col1,Col2,Col3,Col4
Row1,Row1/Col1,Row1/Col2,Row1/Col3,Row1/Col4
Row2,Row2/Col1,Row2/Col2,Row2/Col3,Row2/Col4
Row3,Row3/Col1,Row3/Col2,Row3/Col3,Row3/Col4


## **Data Import**

### Question 1

Import the data file 'student_enrollment_sample.csv' posted with this assignment. This file contains real data from a large online university in the United States. The identifying information has been deleted or randomized to maintain anonymity according to FERPA requirements. 

This university wants to help their students succeed. However, many students drop out for various reasons—some feel hopeless, helpless, or lost. The university wants to identify currently active students who are likely to drop in the near future so they can intervene and provide additional support. Your task is to create a predictive model to classify at-risk students for targeted interventions by the advisement center.

Print the shape of the dataset (rows and columns), then print the first five records.

**Answer this question in the LMS: How many students are in this sample?**

In [ ]:
# Question 1

import pandas as pd

raw = pd.read_csv('student_enrollment_sample.csv')
print(raw.shape)
raw.head()

# **Two-Class Modeling**

## **Data Cleaning**

### Missing Values

### Question 2

Print a list of missing values for each column in the dataset.

HINT: Search for 'pandas .isna() example'

**Answer this question in the LMS: What is the maximum count of missing values in any single column?**

In [ ]:
# Question 2

print(raw.isna().sum())

### Question 3

Make a copy of the original DataFrame to work with (in case you want to use the original again later). Iterate through the new DataFrame and remove any column that has more than 30% of its records missing. Print a summary of missing value percentages for each remaining column.

**Note**: Call this copy `df` and use it for all subsequent data cleaning tasks (Questions 3-7).

**Answer this question in the LMS: What is the percent missing for BIRTH_DATE? Enter the decimal number without converting it to a percent.**

In [ ]:
# Question 3

df = raw.copy()

for col in df.columns:
    if df[col].isna().mean() > 0.30:
        df = df.drop(columns=[col])

print(df.isna().mean())

### Question 4

Drop all remaining rows that contain any missing data. Print the number of rows and columns in the remaining dataset.

**Answer this question in the LMS: How many total columns are left (including both features and labels)?**

In [ ]:
# Question 4

df = df.dropna()
print(df.shape)

### Handle Dates

### Question 5

Now it's time to handle the date values. LAST_ACTIVITY_DATE, EXPECTED_START_DATE, BIRTH_DATE, and GRADUATION_DATE may all be useful. We want to convert these date fields into numeric values.

For LAST_ACTIVITY_DATE, EXPECTED_START_DATE, and BIRTH_DATE, replace each date value with the number of days between that date and 2022-1-1. In other words: 2022-1-1 minus the date value in the field.

For GRADUATION_DATE, replace the date value with the number of days until graduation assuming that today's date is 2020-1-1. In other words, calculate: GRADUATION_DATE minus 2020-1-1.

Print the first five rows to examine the results.

**Answer this question in the LMS: How many days until record 1873 graduates (from the 2020-1-1 reference date)?**

In [ ]:
# Question 5

ref_date = pd.Timestamp('2022-01-01')
grad_ref = pd.Timestamp('2020-01-01')

for col in ['LAST_ACTIVITY_DATE', 'EXPECTED_START_DATE', 'BIRTH_DATE']:
    df[col] = (ref_date - pd.to_datetime(df[col])).dt.days

df['GRADUATION_DATE'] = (pd.to_datetime(df['GRADUATION_DATE']) - grad_ref).dt.days

df.head()

### Bin Categorical Values

### Question 6

The PROGRAM_GROUP feature indicates which academic program the student is enrolled in. Some programs are very small and represent less than 5% of the data. We need to bin those small programs into a new value called "Other".

Start by printing a list of PROGRAM_GROUP values divided by the total number of records to see what percent of cases they represent. Then, use a vectorized approach to change every program value to "Other" if it does not belong to a PROGRAM_GROUP that represents at least 5% of cases. Finally, print the new list of PROGRAM_GROUP values (including the new 'Other') to verify your routine worked correctly. You do not need to print the values in percent format—the original decimal values are fine.

**Answer this question in the LMS: Which program has the most students enrolled? Copy and paste the five-letter acronym.**

In [ ]:
# Question 6

prog_pct = df['PROGRAM_GROUP'].value_counts() / len(df)
print(prog_pct)

keep_programs = prog_pct[prog_pct >= 0.05].index
df.loc[~df['PROGRAM_GROUP'].isin(keep_programs), 'PROGRAM_GROUP'] = 'Other'

print()
print(df['PROGRAM_GROUP'].value_counts() / len(df))

### Relabel Label

### Question 7

Print the first five records of a filtered version of the DataFrame including only the 'IN_SCHOOL_FLAG' and 'STATUS_DESCRIPTION' columns where STATUS_DESCRIPTION equals 'Graduate'. Notice that all graduates have an 'IN_SCHOOL_FLAG' of zero, which makes them appear the same as students who have dropped out or been terminated. Because we want students to graduate, we need to treat graduates the same as active students. 

Convert the 'IN_SCHOOL_FLAG' value for all graduates to 1 (or 1.0). In addition, convert their 'SIMPLE_STATUS_DESCRIPTION' to 'Active'.

After making these changes, calculate and print the proportion of students who are "IN_SCHOOL".

**Answer this question in the LMS: What proportion of students are IN_SCHOOL (IN_SCHOOL_FLAG = 1.0)? Enter the decimal value without converting to a percent.**

In [ ]:
# Question 7

print(df.loc[df['STATUS_DESCRIPTION'] == 'Graduate', ['IN_SCHOOL_FLAG', 'STATUS_DESCRIPTION']].head())

df.loc[df['STATUS_DESCRIPTION'] == 'Graduate', 'IN_SCHOOL_FLAG'] = 1.0
df.loc[df['STATUS_DESCRIPTION'] == 'Graduate', 'SIMPLE_STATUS_DESCRIPTION'] = 'Active'

print()
print(df['IN_SCHOOL_FLAG'].mean())

## **Classification Modeling**

### Dummy Coding

### Question 8

Import the packages necessary for DecisionTreeClassifier and train_test_split. We will use these later. For now, create another copy of the latest DataFrame to work from. Using the new copy, convert MOD_NUMBER and COHORT_YEAR to 'object' data types—these values are numbers, but they represent categorical values (which month and which year). Drop STATUS_DESCRIPTION and SIMPLE_STATUS_DESCRIPTION from the new DataFrame since those are alternative labels and we are going to use IN_SCHOOL_FLAG as the two-class label for our first model. Create dummy codes for all remaining categorical features in the new DataFrame. Print the first five records of the new DataFrame. There should be no remaining categorical values and many new dummy code features.

**Note**: Call this copy `df_model`. This is a separate DataFrame for modeling—you'll still have the original `df` available for the multi-class section later.

**Answer this question in the LMS: How many total columns does df_model have after creating dummy codes? (This represents all features plus the label)**

In [ ]:
# Question 8

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

df_model = df.copy()
df_model['MOD_NUMBER'] = df_model['MOD_NUMBER'].astype('object')
df_model['COHORT_YEAR'] = df_model['COHORT_YEAR'].astype('object')
df_model = df_model.drop(columns=['STATUS_DESCRIPTION', 'SIMPLE_STATUS_DESCRIPTION'])
df_model = pd.get_dummies(df_model)
print(df_model.shape[1])
df_model.head()

### Select Label and Features

### Question 9

Set the y and X variables to represent the label and feature set. Use IN_SCHOOL_FLAG as the label (y) and all other columns as features (X). Print the first five records of the feature list to verify it looks correct.

**Answer this question in the LMS: How many features (columns) are in the X DataFrame?**

In [ ]:
# Question 9

y = df_model['IN_SCHOOL_FLAG']
X = df_model.drop(columns=['IN_SCHOOL_FLAG'])
print(X.shape[1])
X.head()

### Split Data

### Question 10

Split the y and X sets into training and testing sets. Do a 70/30 split, meaning 70% training data. Use a random seed of 12345. Print the shape of the X_train dataset to verify the split.

**Answer this question in the LMS: What is the number of rows in X_train?**

In [ ]:
# Question 10

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=12345)
print(X_train.shape)
X_test.head()

### Create Classifier and Fit Model

### Question 11

Create and fit a DecisionTreeClassifier() model using the training datasets. Use a random_state of 12345 for reproducibility. Print the maximum depth of the trained tree.

**Answer this question in the LMS: What is the maximum depth of the decision tree? This indicates how many levels of splits the tree made.**

In [ ]:
# Question 11

model = DecisionTreeClassifier(random_state=12345)
model.fit(X_train, y_train)
print(model.tree_.max_depth)

### Compare Actual Versus Predicted Values

### Question 12

Generate predictions for the testing dataset. Add the predicted values to a new DataFrame along with the actual values and print the first 10 records.

**Answer this question in the LMS: How many of the first 10 records/predictions are inaccurate?**

In [ ]:
# Question 12

y_pred = model.predict(X_test)
results = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred})
print(results.head(10))

### Assess Model Fit/Quality/Accuracy

### Question 13

Generate a confusion matrix for the results. Use display_labels of ['Quit', 'Active'] to make the matrix more interpretable.

**Answer this question in the LMS: How many students are currently active (actual = 1.0) but predicted to quit (predicted = 0.0)?**

In [ ]:
# Question 13

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Quit', 'Active'])
plt.show()

### Question 14

Generate the accuracy, precision, recall, and F1 scores for the predictions. For precision, recall, and F1, specify that we're interested in predicting the "Active" class (pos_label=1.0).

**Answer this question in the LMS: What is the accuracy score? Do not convert it to a percent or round it.**

In [ ]:
# Question 14

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label=1.0))
print("Recall:", recall_score(y_test, y_pred, pos_label=1.0))
print("F1:", f1_score(y_test, y_pred, pos_label=1.0))

### Visualize Classification Model

### Question 15

Generate a tree visualization using export_graphviz.

**Answer this question in the LMS: Which feature is most important in determining IN_SCHOOL_FLAG? Enter the name exactly as it appears in the dataset (including underscores and capitalization).**

In [ ]:
# Question 15

from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(model, out_file=None, feature_names=X.columns, filled=True, rounded=True, max_depth=3)
graphviz.Source(dot_data)

# **Multi-Class Modeling**

## **Data Cleaning**

### Combine or Eliminate Low Frequency Values

### Question 16

The label 'IN_SCHOOL_FLAG' was ideal for a two-class classification model. However, most students are put on probation before they are terminated. Therefore, it could be useful to predict three distinct categories: Active, Probation, and Terminated. First, let's identify what label combinations exist across all three label-related columns. Print a list of all value combinations of 'STATUS_DESCRIPTION', 'IN_SCHOOL_FLAG', and 'SIMPLE_STATUS_DESCRIPTION' along with a count of how often each occurs.

**Note**: Use the original `df` DataFrame (not `df_model`) for this and the following multi-class preparation questions (16-19).

HINT: Consider using the .groupby() method of pandas to perform this in a single line of code. The purpose of this task is to explore your options for potential labels in this dataset. Each of these features represents current states for each student.

**Answer this question in the LMS: How many unique STATUS_DESCRIPTION values exist in the output table? This helps us understand the complexity of student statuses.**

In [ ]:
# Question 16

print(df.groupby(['STATUS_DESCRIPTION', 'IN_SCHOOL_FLAG', 'SIMPLE_STATUS_DESCRIPTION'])['ENROLL_COUNT'].count())

### Question 17

To simplify our eventual model, eliminate all records for students whose 'SIMPLE_STATUS_DESCRIPTION' equals 'Other'. Then, reprint the table above to ensure it worked properly.

**Answer this question in the LMS: After filtering out 'Other', how many unique combinations of STATUS_DESCRIPTION, IN_SCHOOL_FLAG, and SIMPLE_STATUS_DESCRIPTION remain?**

In [ ]:
# Question 17

df = df[df['SIMPLE_STATUS_DESCRIPTION'] != 'Other']
print(df.groupby(['STATUS_DESCRIPTION', 'IN_SCHOOL_FLAG', 'SIMPLE_STATUS_DESCRIPTION'])['ENROLL_COUNT'].count())

### Question 18

Next, eliminate anyone whose 'STATUS_DESCRIPTION' equals 'Transfer To Other Program', 'No Show', or 'False Start' since those outcomes are not relevant to students who were once active. Reprint the same summary table afterward.

**Answer this question in the LMS: After removing these three status types, how many total students remain in the dataset?**

In [ ]:
# Question 18

df = df[~df['STATUS_DESCRIPTION'].isin(['Transfer To Other Program', 'No Show', 'False Start'])]
print(df.groupby(['STATUS_DESCRIPTION', 'IN_SCHOOL_FLAG', 'SIMPLE_STATUS_DESCRIPTION'])['ENROLL_COUNT'].count())

### Question 19

Finally, relabel the 'SIMPLE_STATUS_DESCRIPTION' of those whose 'STATUS_DESCRIPTION' equals 'Probation' to 'Probation'. This will create our three-class label: Active, Probation, and Terminated. Reprint the table.

**Answer this question in the LMS: How many students now have 'Probation' as their SIMPLE_STATUS_DESCRIPTION? This is our new third class for the multi-class model.**

In [ ]:
# Question 19

df.loc[df['STATUS_DESCRIPTION'] == 'Probation', 'SIMPLE_STATUS_DESCRIPTION'] = 'Probation'
print(df.groupby(['STATUS_DESCRIPTION', 'IN_SCHOOL_FLAG', 'SIMPLE_STATUS_DESCRIPTION'])['ENROLL_COUNT'].count())

## **Classification Modeling**

### Dummy Codes

### Question 20

Create a new copy of the DataFrame to work from. The features 'MOD_NUMBER' and 'COHORT_YEAR' represent categorical values that do not have an inherent order. Therefore, they should be treated as categories and cast to objects. Once you have done so, drop the alternative labels 'STATUS_DESCRIPTION' and 'IN_SCHOOL_FLAG' from the dataset because we will be predicting 'SIMPLE_STATUS_DESCRIPTION'. Then, generate dummy codes for the remaining categorical features (but NOT for SIMPLE_STATUS_DESCRIPTION since that's our label). Print the first five records to verify everything worked correctly.

**Note**: Call this copy `df_model` (yes, you can reuse this variable name from the two-class section). This creates a fresh modeling DataFrame for the multi-class model.

**Answer this question in the LMS: How many columns does df_model have after creating dummy codes? (This includes both features and the SIMPLE_STATUS_DESCRIPTION label)**

In [ ]:
# Question 20

df_model = df.copy()
df_model['MOD_NUMBER'] = df_model['MOD_NUMBER'].astype('object')
df_model['COHORT_YEAR'] = df_model['COHORT_YEAR'].astype('object')
df_model = df_model.drop(columns=['STATUS_DESCRIPTION', 'IN_SCHOOL_FLAG'])

label_col = df_model['SIMPLE_STATUS_DESCRIPTION']
features = df_model.drop(columns=['SIMPLE_STATUS_DESCRIPTION'])
features = pd.get_dummies(features)
df_model = pd.concat([features, label_col], axis=1)

print(df_model.shape[1])
df_model.head()

### Select Label and Features

### Question 21

Create the y and X variables to store the label and features using 'SIMPLE_STATUS_DESCRIPTION' as the label. Print the unique values in y to verify the three classes.

**Answer this question in the LMS: How many unique classes are in y?**

In [ ]:
# Question 21

y = df_model['SIMPLE_STATUS_DESCRIPTION']
X = df_model.drop(columns=['SIMPLE_STATUS_DESCRIPTION'])
print(y.unique())

### Split Data

### Question 22

Split the data using a 70/30 split and 12345 as the random seed. Print the shape of y_train to verify the split.

**Answer this question in the LMS: What is the number of samples in y_train? This represents 70% of the total dataset after filtering.**

In [ ]:
# Question 22

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=12345)
print(y_train.shape)

### Create Classifier and Fit Model

### Question 23

Train a decision tree classifier model using the datasets you just generated. Use random_state=12345 for reproducibility. Print the number of classes the model learned. You may need to ask AI or Google to help you learn how to print the number of classes learned.

**Answer this question in the LMS: How many classes did the model learn?**

In [ ]:
# Question 23

model = DecisionTreeClassifier(random_state=12345)
model.fit(X_train, y_train)
print(model.n_classes_)

### Compare Actual Versus Predicted Values

### Question 24

Predict the y values for the testing dataset and add them to a DataFrame along with the actual y values for comparison. Print the first 20 records.

**Answer this question in the LMS: How many of these first 20 predictions were inaccurate?**

In [ ]:
# Question 24

y_pred = model.predict(X_test)
results = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred})
print(results.head(20))

### Assess Model Fit/Quality/Accuracy

### Question 25

Generate and print a confusion matrix to view the results. The confusion matrix will show how well the model distinguishes between Active, Probation, and Terminated students.

**Answer this question in the LMS: In a multi-class confusion matrix, which class is the model best at predicting correctly?**

In [ ]:
# Question 25

from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()

### Question 26

Generate the fit metrics (accuracy, precision, recall, F1) for the multi-class model. Use 'weighted' averaging to account for class imbalance, which gives each class's metric a weight based on how many samples are in that class.

**Answer this question in the LMS: What is the weighted precision score? Enter the full decimal value without rounding. This metric tells us what proportion of students predicted for each class actually belong to that class.**

In [ ]:
# Question 26

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1:", f1_score(y_test, y_pred, average='weighted'))

### Visualize Classification Model

### Question 27

Generate a decision tree visualization using export_graphviz.

**Answer this question in the LMS: Which feature is most important in this multi-class model predicting 'SIMPLE_STATUS_DESCRIPTION' if MODS_ATTENDED_COUNT is less than or equal to 4.5? Enter the name exactly as it appears including underscores and capitalization.**

In [ ]:
# Question 27

dot_data = export_graphviz(model, out_file=None, feature_names=X.columns, filled=True, rounded=True, max_depth=3)
graphviz.Source(dot_data)